In [ ]:
import os 
import ast
import numpy as np
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
from wquantiles import quantile
from statsmodels.stats.weightstats import DescrStatsW
from openfisca_france_indirect_taxation import FranceIndirectTaxationTaxBenefitSystem
from openfisca_france_indirect_taxation.surveys import SurveyScenario
from openfisca_france_indirect_taxation.utils import assets_directory, get_input_data_frame
from openfisca_france_indirect_taxation.projects.TVA.Utils import weighted_quantiles 
from openfisca_france_indirect_taxation.build_survey_data.utils import collapsesum
from openfisca_france_indirect_taxation.Calage_revenus_bdf import compute_erfs_decile, calage_bdf_niveau_vie
from openfisca_france_indirect_taxation.Calage_consommation_bdf import get_inflators_by_year
from openfisca_france_indirect_taxation.examples.utils_example import df_weighted_average_grouped, wavg, collapse


In [ ]:
from openfisca_core.taxbenefitsystems import TaxBenefitSystem

In [ ]:
year = 2018
data_year = 2017
tax_benefit_system = FranceIndirectTaxationTaxBenefitSystem()
inflators_by_year = get_inflators_by_year(rebuild = False, year_range = range(2017, 2025), data_year = data_year)
inflation_kwargs = dict(inflator_by_variable = inflators_by_year[year])

inflation_kwargs.get('inflator_by_variable').update({'depenses_carburants_entree' : inflation_kwargs.get('inflator_by_variable').pop('depenses_carburants'),
                                                     'depense_gazole_total_ttc_entree' : inflation_kwargs.get('inflator_by_variable').pop('depenses_diesel'),
                                                     'depense_essence_total_ttc_entree' : inflation_kwargs.get('inflator_by_variable').pop('depenses_essence'),})

In [ ]:
input_bdf = get_input_data_frame(2017, use_emp = True)
input_bdf.rename({'depenses_carburants' : 'depenses_carburants_entree',
                  'depenses_diesel' : 'depense_gazole_total_ttc_entree',
                  'depenses_essence' : 'depense_essence_total_ttc_entree'}, axis = 1, inplace= True)
input_bdf = input_bdf.loc[input_bdf['rev_disponible'] > 0]

erfs_path = 'C:/Users/veve1/OneDrive/Documents/ENSAE 3A/Memoire MiE/Data/erfs_fpr/{}/csv'.format(year) 
erfs_menage_by_decile = compute_erfs_decile(year, 'men', erfs_path)
erfs_menage_by_decile

input_bdf , df_calage = calage_bdf_niveau_vie(input_bdf, erfs_menage_by_decile, 'men')

In [ ]:
simulated_variables = [
    'distance',
    'veh_tot',
    'nb_diesel',
    'nactifs',
    'paris',
    'rural',
    'depenses_carburants_entree',
    'depense_essence_total_ttc_entree', 
    'depense_gazole_total_ttc_entree',
    'depense_essence_total_ttc',
    'depense_gazole_total_ttc',
    'depense_carburant_total_ttc',
    'depense_essence_total_ht',
    'depense_gazole_total_ht',
    'depense_carburant_total_ht',
    'essence_ticpe_total',
    'gazole_ticpe_total',
    'ticpe_carburant_total',
    'tva_sur_essence_total',
    'tva_sur_gazole_total', 
    'tva_sur_carburant_total',
    'taxes_essence_total',
    'taxes_gazole_total',
    'taxes_carburant_total',
    'emissions_CO2_carburants',
    'cheques_energie',
    'niveau_vie_decile', 'pondmen', 'npers',
    'rev_disponible',
    'ocde10', 
    'niveau_de_vie'
]

In [ ]:
survey_scenario = SurveyScenario.create(
    input_data_frame = input_bdf,   # Les niveaux de vie sont calés sur ceux de l'ERFS 2018 (pour des déciles d'individus !)
    inflation_kwargs =  inflation_kwargs,
    tax_benefit_system = tax_benefit_system,
    period = year,
    )

baseline_menage = survey_scenario.create_data_frame_by_entity(simulated_variables, use_baseline = True, period = year)['menage']

In [ ]:
stats_varlist = [
        'npers',
        'nactifs',
        'veh_tot',
        'paris',
        'rural',
        'distance',
        'depenses_carburants_entree',
        'rev_disponible']

stats = {'Mean' : {}, 'Std' : {}}
stats['Mean'] = dict(baseline_menage.loc[:, stats_varlist].apply(lambda x : round(DescrStatsW(x, baseline_menage['pondmen']).mean, 1)))
stats['Std'] = dict(baseline_menage.loc[:, stats_varlist].apply(lambda x : round(DescrStatsW(x, baseline_menage['pondmen']).std, 1)))

stats_desc = pd.DataFrame(stats).rename(index = {
    'npers' : 'Number of person in household',
    'nactifs' : 'Number of active persons',
    'veh_tot' : 'Number of vehicles',
    'paris' : 'Living in Paris',
    'rural' : 'Living in rural area',
    'distance' : 'Annual distance traveled (km)',
    'depenses_carburants_entree' : 'Fuel expenses (€)',
    'rev_disponible' : 'Disposable income (€)'
})
stats_desc

In [ ]:
stats_desc_by_decile = df_weighted_average_grouped(baseline_menage,
                            'niveau_vie_decile',
                            stats_varlist ,
                            'pondmen')

In [ ]:
pd.options.display.float_format = '{:.2f}'.format

In [ ]:
stats_desc_by_decile.rename(columns= { 'npers' : '# person in household',
    'nactifs' : '# active persons',
    'veh_tot' : '# vehicles',
    'paris' : 'Paris',
    'rural' : 'Rural area',
    'distance' : 'Annual distance traveled (km)',
    'depenses_carburants_entree' : 'Fuel expenses (€)',
    'rev_disponible' : 'Disposable income (€)'})

In [ ]:
output_path = "C:/Users/veve1/OneDrive/Documents/ENSAE PhD/Carbon tax/Output/Figures/Reform"

In [ ]:
DIESEL_KG_CO2_PAR_HL = 264.5
ESSENCE_KG_CO2_PAR_HL = 228.4
ETHANOL_KG_CO2_PAR_HL = 171.6

from openfisca_core.reforms import Reform

class reform_ticpe_2019_in_2018(Reform):
    name = u'Augmentation de la TICPE programmée pour 2019'

    def apply(self):
        def reform_modify_parameters(parameters):
            reform_parameters = parameters
            reform_parameters.imposition_indirecte.produits_energetiques.ticpe.gazole.update(
                start = "2018-01-01",
                value = 59.4 + 2.6 + (55.0 - 44.6) / 1e3 * DIESEL_KG_CO2_PAR_HL
            )
            reform_parameters.imposition_indirecte.produits_energetiques.ticpe.super_95_98.update(
                start = "2018-01-01",
                value = 68.29 + (55.0 - 44.6) / 1e3 * ESSENCE_KG_CO2_PAR_HL
            )
            reform_parameters.imposition_indirecte.produits_energetiques.ticpe.super_e10.update(
                start = "2018-01-01",
                value = 66.29 + (55.0 - 44.6) / 1e3 * ESSENCE_KG_CO2_PAR_HL
            )
            reform_parameters.imposition_indirecte.produits_energetiques.ticpe.super_plombe.update(
                start = "2018-01-01",
                value = 71.56 + (55.0 - 44.6) / 1e3 * ESSENCE_KG_CO2_PAR_HL
            )
            reform_parameters.imposition_indirecte.produits_energetiques.ticpe.super_e_85_utilise_comme_carburant_hectolitre.update(
                start = "2018-01-01",
                value = 11.83 + (55.0 - 44.6) / 1e3 * ESSENCE_KG_CO2_PAR_HL
            )
            reform_parameters.imposition_indirecte.emissions_CO2.carburants.CO2_diesel.update(
                start = "2018-01-01",
                value = DIESEL_KG_CO2_PAR_HL / 100
            )
            reform_parameters.imposition_indirecte.emissions_CO2.carburants.CO2_essence.update(
                start = "2018-01-01",   
                value = ESSENCE_KG_CO2_PAR_HL / 100
            )
            return parameters

        self.modify_parameters(modifier_function = reform_modify_parameters)

In [ ]:
varlist = ['depense_essence_total_ttc',
    'depense_gazole_total_ttc',
    'depense_carburant_total_ttc',
    'depense_essence_total_ht',
    'depense_gazole_total_ht',
    'depense_carburant_total_ht',
    'essence_ticpe_total',
    'gazole_ticpe_total',
    'ticpe_carburant_total',
    'tva_sur_essence_total',
    'tva_sur_gazole_total', 
    'tva_sur_carburant_total',
    'taxes_essence_total',
    'taxes_gazole_total',
    'taxes_carburant_total',
    'emissions_CO2_carburants',
    'rev_disponible', 
    'redistribution_reform',
    'emissions_CO2_carburants',
    'emissions_CO2_carburants_baseline',
    'npers',
    'veh_tot',]

In [ ]:
def simulate_reform_ticpe_2019_in_2018(input_bdf, year, ref_elasticity):    
    
    survey_scenario = SurveyScenario.create(
        input_data_frame = input_bdf,           # Les niveaux de vie sont calés sur ceux de l'ERFS 2018 (pour des déciles d'individus !)
        inflation_kwargs =  inflation_kwargs,
        baseline_tax_benefit_system = tax_benefit_system,
        reform = reform_ticpe_2019_in_2018,
        period = year,
        )

    baseline_menage = survey_scenario.create_data_frame_by_entity(simulated_variables, use_baseline = True, period = year)['menage']
    reform_menage = survey_scenario.create_data_frame_by_entity(simulated_variables, use_baseline = False, period = year)['menage']

    # On calcule la différence entre la situation de référence et la situation avec réforme pour les variables d'intérêt
    difference_menage = baseline_menage - reform_menage
    variables = ['rev_disponible','niveau_de_vie','niveau_vie_decile','ocde10','pondmen', 'npers', 'veh_tot']
    difference_menage.loc[:, variables ] = baseline_menage.loc[:, variables]
    difference_menage.loc[:, 'ref_elasticity'] = ref_elasticity
    difference_menage['emissions_CO2_carburants_baseline'] = baseline_menage['emissions_CO2_carburants']
     
    # On calcule la redistribution du produit de la réforme (uniquement TICPE pas TVA)
    delta_ticpe_total = survey_scenario.compute_aggregate('ticpe_carburant_total', aggfunc='sum', difference= True, period = year)
    pondmen_total = survey_scenario.compute_aggregate('pondmen', aggfunc='sum', use_baseline= True, weighted= False, period = year)
    difference_menage.loc[:, 'redistribution_reform'] = delta_ticpe_total / pondmen_total

    difference_menage['taxes_carburant_sur_revenu'] = difference_menage['taxes_carburant_total'] / difference_menage['rev_disponible'] * 100
    difference_menage.loc[:, 'Net_transfer_reform'] = difference_menage.loc[:, 'taxes_carburant_total'] + difference_menage.loc[:, 'redistribution_reform']
    difference_menage.loc[:, 'Net_transfer_sur_revenu'] = difference_menage.loc[:, 'Net_transfer_reform'] / difference_menage.loc[:, 'rev_disponible'] * 100
    difference_menage.loc[:, 'pct_reduction_CO2_emissions'] = difference_menage.loc[:, 'emissions_CO2_carburants'] / difference_menage.loc[:, 'emissions_CO2_carburants_baseline'] * 100
    
    # On crée une dataframe avec les moyennes pondérées par déciles d'individus (difference = True)
    difference_decile = df_weighted_average_grouped(difference_menage,
                                                    'niveau_vie_decile',
                                                    varlist ,
                                                    'pondmen')

    difference_decile['pondmen'] = difference_menage.groupby('niveau_vie_decile')['pondmen'].sum()
    
    difference_decile['taxes_carburant_sur_revenu'] = difference_decile['taxes_carburant_total'] / difference_decile['rev_disponible'] * 100
    difference_decile.loc[:, 'Net_transfer_reform'] = difference_decile.loc[:, 'taxes_carburant_total'] + difference_decile.loc[:, 'redistribution_reform']
    difference_decile.loc[:, 'Net_transfer_sur_revenu'] = difference_decile.loc[:, 'Net_transfer_reform'] / difference_decile.loc[:, 'rev_disponible'] * 100
    difference_decile.loc[:, 'pct_reduction_CO2_emissions'] = - difference_decile.loc[:, 'emissions_CO2_carburants'] / difference_decile.loc[:, 'emissions_CO2_carburants_baseline'] * 100
    
    difference_decile['Avg_taxes_carburant_total'] = wavg(difference_decile, 'taxes_carburant_total', 'pondmen')
    difference_decile['Avg_taxes_carburant_revenu'] =  difference_decile['Avg_taxes_carburant_total'] / wavg(difference_decile, 'rev_disponible', 'pondmen') * 100
    
    difference_decile['Avg_net_transfer_reform'] = wavg(difference_decile, 'Net_transfer_reform', 'pondmen')
    difference_decile['Avg_net_transfer_revenu'] = difference_decile['Avg_net_transfer_reform'] / wavg(difference_decile, 'rev_disponible', 'pondmen') * 100
    
    difference_decile['Avg_emission_reduction'] = - wavg(difference_decile, 'emissions_CO2_carburants', 'pondmen')
    difference_decile['Avg_emission_baseline'] = wavg(difference_decile, 'emissions_CO2_carburants_baseline', 'pondmen')
    difference_decile['Avg_pct_reduction_CO2_emissions'] = difference_decile['Avg_emission_reduction'] / difference_decile['Avg_emission_baseline'] * 100
    
    difference_decile.loc[:, 'ref_elasticity'] = ref_elasticity
    
    df_sum = pd.DataFrame()
    df_sum['delta_emissions_C02_carburants'] = - collapsesum(difference_menage, 'niveau_vie_decile', 'emissions_CO2_carburants', 'pondmen')
    df_sum['emissions_CO2_carburants_baseline'] = collapsesum(baseline_menage, 'niveau_vie_decile', 'emissions_CO2_carburants', 'pondmen')

    df_sum['ratio_emissions_reductions'] = df_sum['delta_emissions_C02_carburants'] / df_sum['emissions_CO2_carburants_baseline'] * 100
    df_sum['total_emissions_baseline'] = df_sum['emissions_CO2_carburants_baseline'].sum()
    df_sum['total_emissions_reductions'] = df_sum['delta_emissions_C02_carburants'].sum()
    df_sum['share_emissions'] = df_sum['emissions_CO2_carburants_baseline'] / df_sum['total_emissions_baseline'] * 100
    df_sum['share_emissions_reductions'] = df_sum['delta_emissions_C02_carburants'] / df_sum['total_emissions_reductions'] * 100
    df_sum['ratio_share_share'] = df_sum['share_emissions_reductions'] / df_sum['share_emissions']
    df_sum.loc[:, 'ref_elasticity'] = ref_elasticity
    
    return difference_decile, difference_menage, df_sum


In [ ]:
data_path = "C:/Users/veve1/OneDrive/Documents/ENSAE 3A/Memoire MiE/Data"
elasticities = pd.read_csv(os.path.join(data_path, 'Reform_parameters/Elasticities_Douenne_20.csv'), usecols=['niveau_vie_decile', 'ref_elasticity', 'Rural', 'Small cities', 'Medium cities', 'Large cities', 'Paris'])
elasticities = pd.melt(frame = elasticities, id_vars = ["niveau_vie_decile", 'ref_elasticity'], var_name = 'strate_2', value_name = 'elas_price_1_1')

In [ ]:
# difference_menage = pd.DataFrame(columns = simulated_variables + ['ref_elasticity', 'Net_transfer_reform', 'Net_transfer_sur_revenu', 'pondindiv', 'redistribution_reform'])
# difference_decile = pd.DataFrame(columns = varlist + ['ref_elasticity', 'Avg_net_transfer_reform', 'Avg_net_transfer_revenu', 'Net_transfer_reform', 'Net_transfer_sur_revenu',
#     'pondindiv', 'redistribution_reform', 'rev_disponible'])                          

# Homogeneosu Douenne (2020) elasiticity
input_bdf['elas_exp_1'] = - 0.45 
difference_decile, difference_menage, df_sum = simulate_reform_ticpe_2019_in_2018(input_bdf, year, 'Douenne (2020)')

# Elasticities by income decile and location 
dict_strate = {0: 'Rural', 1: 'Small cities', 2: 'Medium cities', 3: 'Large cities', 4: 'Paris'}
input_bdf['strate_2'] = input_bdf['strate'].apply(lambda x: dict_strate.get(x))
input_bdf['niveau_de_vie'] = (input_bdf['rev_disponible'] / input_bdf['ocde10']).astype(float)
input_bdf['pondmen'] = input_bdf['pondmen'].astype(int)
input_bdf['niveau_vie_decile'] = weighted_quantiles(input_bdf['niveau_de_vie'], labels = np.arange(1, 11), weights = input_bdf['pondmen'])
input_bdf = input_bdf.merge(right = elasticities, how = 'inner', on = ['niveau_vie_decile', 'strate_2'])
input_bdf['elas_exp_1'] = input_bdf['elas_price_1_1'].apply(lambda x: ast.literal_eval(x)[0])
input_bdf.drop('elas_price_1_1', axis = 1, inplace = True)
to_concat = simulate_reform_ticpe_2019_in_2018(input_bdf, year, 'Douenne (2020) with heterog.')
difference_decile = pd.concat([difference_decile, to_concat[0]])
difference_menage = pd.concat([difference_menage, to_concat[1]])
df_sum = pd.concat([df_sum, to_concat[2]])

In [ ]:
hue_order = ['Douenne (2020)', 'Douenne (2020) with heterog.']

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.barplot(data = difference_decile.loc[difference_decile['ref_elasticity'] == 'Douenne (2020)'], x='niveau_vie_decile', y='npers', color = sns.color_palette("Paired")[0])
plt.xlabel('Equivalised income decile', size = 16)
plt.ylabel('Number of persons in households', size = 16)
plt.xticks(fontsize = 16)
plt.yticks(fontsize = 16)
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig(os.path.join(output_path,'Nper_per_decile.pdf'), bbox_inches = 'tight')

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.barplot(data = difference_decile.loc[difference_decile['ref_elasticity'] == 'Douenne (2020)'], x='niveau_vie_decile', y='veh_tot', color = sns.color_palette("Paired")[0])
plt.xlabel('Equivalised income decile', size = 16)
plt.ylabel('Number of vehicles in households', size = 16)
plt.xticks(fontsize = 16)
plt.yticks(fontsize = 16)
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig(os.path.join(output_path,'Veh_tot_per_decile.pdf'), bbox_inches = 'tight')

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.barplot(data = difference_decile, x='niveau_vie_decile', y='taxes_carburant_total', hue = 'ref_elasticity', hue_order= hue_order, palette = sns.color_palette("Paired",2 ))
mean_taxes_carburant= difference_decile['Avg_taxes_carburant_total'].iloc[0]
plt.axhline(y = mean_taxes_carburant, ls = '--', color = 'orange', linewidth = 2.5)
plt.text(
    x = 2, 
    y = mean_taxes_carburant ,
    s = f'Mean = {mean_taxes_carburant:.1f} €',  # Texte à afficher
    ha = 'right',  # Alignement horizontal
    va = 'bottom',  # Alignement vertical
    fontsize= 16,
    color = 'orange'
)
plt.xlabel('Equivalised income decile', size = 16)
plt.ylabel('Net effect per household (in €)', size = 16)
plt.xticks(fontsize = 16)
plt.yticks(  fontsize = 16)
plt.legend(loc = 'lower left', fontsize = 16)
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig(os.path.join(output_path,'Additional_taxes.pdf'), bbox_inches = 'tight')

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.barplot(data = difference_decile, x='niveau_vie_decile', y='taxes_carburant_sur_revenu', hue = 'ref_elasticity', hue_order= hue_order, palette = sns.color_palette("Paired",2 ))
mean_taxes_carburant= difference_decile['Avg_taxes_carburant_revenu'].iloc[0]
plt.axhline(y = mean_taxes_carburant, ls = '--', color = 'orange', linewidth = 2.5)
plt.text(
    x = 9.1, 
    y = mean_taxes_carburant - 0.015,
    s = f'Mean = {mean_taxes_carburant:.2f} %',  # Texte à afficher
    ha = 'right',  # Alignement horizontal
    va = 'bottom',  # Alignement vertical
    fontsize= 16,
    color = 'orange'
)
plt.xlabel('Equivalised income decile', size = 16)
plt.ylabel('Additional taxes (% of disp income)', size = 16)
plt.xticks(fontsize = 16)
plt.yticks(  fontsize = 16)
plt.legend(loc = 'lower right', fontsize = 16)
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig(os.path.join(output_path,'Additional_taxes_sur_revenu.pdf'), bbox_inches = 'tight')

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.barplot(data = difference_decile, x='niveau_vie_decile', y='Net_transfer_reform', hue = 'ref_elasticity', hue_order= hue_order, palette = sns.color_palette("Paired",2 ))
mean_transfer_reform = difference_decile['Avg_net_transfer_reform'].iloc[0]
plt.axhline(y = mean_transfer_reform, ls = '--', color = 'orange', linewidth = 2.5)
plt.text(
    x = 2, 
    y = mean_transfer_reform ,
    s = f'Mean = {mean_transfer_reform:.1f} €',  # Texte à afficher
    ha = 'right',  # Alignement horizontal
    va = 'bottom',  # Alignement vertical
    fontsize= 16,
    color = 'orange'
)
plt.xlabel('Equivalised income decile', size = 16)
plt.ylabel('Net effect per household (in €)', size = 16)
plt.xticks(fontsize = 16)
plt.yticks(np.arange(-25,30,5),  fontsize = 16)
plt.legend(loc = 'upper right', fontsize = 16)
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig(os.path.join(output_path,'Net_transfer_reform.pdf'), bbox_inches = 'tight')

In [ ]:
net_transfer_reform_boxplot = quantiles_for_boxplot(difference_menage, 'Net_transfer_reform', 'pondmen')

In [ ]:
plt.figure(figsize=(10, 6))
plt.axhline(y = 0, ls = '--', color = 'black', linewidth = 1)
ax = sns.scatterplot(data = net_transfer_reform_boxplot, x='plot_decile', y='Net_transfer_reform',  
                     hue = 'ref_elasticity', hue_order= hue_order, palette = sns.color_palette("Paired",2 ),
                    style = 'Percentile',
                    markers = ['v', 'd', 'o', 'd', '^'],
                    s = 100,
                    legend = True)
plt.xlabel('Equivalised income decile', size = 16)
plt.ylabel('Net transfers per households (in €)', size = 16)
plt.xticks(np.arange(1,11), fontsize = 16)
plt.yticks([-100, -60, -20, 0, 20, 60, 100], fontsize = 16)
plt.grid(True, linestyle='--', alpha=0.7)
labels, handles = ax.get_legend_handles_labels() 
ax.legend(labels[4:], handles[4:], loc='upper left', 
          fancybox=True, ncol=5, fontsize = 14, columnspacing =0)
plt.savefig(os.path.join(output_path,'Box_plot_net_transfer_reform.pdf'), bbox_inches = 'tight')

In [ ]:
net_transfer_revenu_boxplot = quantiles_for_boxplot(difference_menage, 'Net_transfer_sur_revenu', 'pondmen')

In [ ]:
net_transfer_reform_boxplot

In [ ]:
plt.figure(figsize=(10, 6))
plt.axhline(y = 0, ls = '--', color = 'black', linewidth = 1)
ax = sns.scatterplot(data = net_transfer_revenu_boxplot, x='plot_decile', y='Net_transfer_sur_revenu', hue = 'ref_elasticity', hue_order= hue_order, palette = sns.color_palette("Paired",2 ),
                style = 'Percentile',
                markers = ['v', 'd', 'o', 'd', '^'],
                s = 100,
                legend = True)
plt.xlabel('Equivalised income decile', size = 16)
plt.ylabel('Net transfers per households \n (in % of disp income)', size = 16)
plt.xticks(np.arange(1,11), fontsize = 16)
plt.yticks(np.arange(-0.4, 1.2, 0.2), fontsize = 16)
plt.grid(True, linestyle='--', alpha=0.7)
labels, handles = ax.get_legend_handles_labels() 
ax.legend(labels[4:], handles[4:], loc='upper left', 
          fancybox=True, ncol=5, fontsize = 14, columnspacing =0)
plt.savefig(os.path.join(output_path,'Box_plot_net_transfer_revenu.pdf'), bbox_inches = 'tight')

In [ ]:
difference_decile['emissions_CO2_carburants_baseline'] = difference_decile['emissions_CO2_carburants_baseline'] / 1E3
difference_decile['Avg_emission_baseline'] = difference_decile['Avg_emission_baseline'] / 1E3

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.barplot(data = difference_decile.loc[difference_decile['ref_elasticity'] == 'Douenne (2020)'], x='niveau_vie_decile', y='emissions_CO2_carburants_baseline', color = sns.color_palette("Paired")[0])
mean_emission = difference_decile['Avg_emission_baseline'].iloc[0]
plt.axhline(y = mean_emission, ls = '--', color = 'orange', linewidth = 2.5)
plt.text(
    x = 2, 
    y = 3 ,
    s = f'Mean = {mean_emission:.1f} tons',  # Texte à afficher
    ha = 'right',  # Alignement horizontal
    va = 'bottom',  # Alignement vertical
    fontsize= 16,
    color = 'orange'
)
plt.xlabel('Equivalised income decile', size = 16)
plt.ylabel('Baseline CO2 emissions (in tons)', size = 16)
plt.xticks(fontsize = 16)
plt.yticks(fontsize = 16)
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig(os.path.join(output_path,'Total_emissions_baseline.pdf'), bbox_inches = 'tight')

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.barplot(data = difference_decile, x='niveau_vie_decile', y='pct_reduction_CO2_emissions', hue = 'ref_elasticity', hue_order= hue_order, palette = sns.color_palette("Paired",2 ))
mean_pct_reduction = difference_decile['Avg_pct_reduction_CO2_emissions'].iloc[0]
plt.axhline(y = mean_pct_reduction, ls = '--', color = 'orange', linewidth = 2.5)
plt.text(
    x = 2, 
    y =-4 ,
    s = f'Mean = {mean_pct_reduction:.1f} %',  # Texte à afficher
    ha = 'right',  # Alignement horizontal
    va = 'bottom',  # Alignement vertical
    fontsize= 16,
    color = 'orange'
)
plt.xlabel('Equivalised income decile', size = 16)
plt.ylabel('Reduction in CO2 emissions (in %)', size = 16)
plt.xticks(fontsize = 16)
plt.yticks(np.arange(-5, 1, 1), fontsize = 16)
plt.legend(loc = 'lower right', fontsize = 16)
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig(os.path.join(output_path,'Average_emissions_reduction.pdf'), bbox_inches = 'tight')

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.barplot(data = df_sum, x='niveau_vie_decile', y='ratio_share_share', hue = 'ref_elasticity', hue_order= hue_order, palette = sns.color_palette("Paired",2 ))
plt.axhline(y = 1, ls = '--', color = 'black', linewidth = 1)
plt.xlabel('Equivalised income decile', size = 16)
plt.ylabel('Share of total emissions reduction \n over share of total emissions', size = 16)
plt.xticks(fontsize = 16)
plt.yticks(np.arange(0, 2, 0.5), fontsize = 16)
plt.legend(loc = 'upper right', fontsize = 16)
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig(os.path.join(output_path,'Share_reduction_over_share_emissions.pdf'), bbox_inches = 'tight')